<a href="https://colab.research.google.com/github/RicUnil/Datascience_Advanced_Programming_Project/blob/main/SAAM_Project_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SAAM Project 2026 - Portfolio Allocation with a Carbon Objective
Group AI


# PART 0 - SETUP & IMPORTS

Import :   
DS_CO2_SCOPE_1_Y_2025.xlsx  
DS_MV_T_USD_M_2025.xlsx  
DS_MV_T_USD_Y_2025.xlsx  
DS_REV_Y_2025.xlsx  
DS_RI_T_USD_M_2025.xlsx  
DS_RI_T_USD_Y_2025.xlsx    
Risk_Free_Rate_2025.xlsx  
Static_2025.xlsx  

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import time
import cvxpy as cp
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from google.colab import files

files.upload()

data_path = ""

### 0.1 Project parameters
All project parameters in one place to make easy to change and rerun

In [ ]:
target_regions = ["AMER", "EUR"]
scope_used = "Scope 1"
first_allocation_year = 2013
last_allocation_year = 2024
first_perf_year = 2014
last_perf_year = 2025
estimation_window_months = 120
minimum_valid_months = 36
maximum_zero_return_share = 0.50
minimum_price  = 0.5
portfolio_value = 1.0 #millions USD
net_zero_reduction_rate = 0.10
carbon_reduction_target = 0.50

## 1. Data Loading and Cleaning

### 1.1 Load the static file
This first step loads the static file, which gives the main information about each company:ISIN, name, country, and region. The ISIN allows us to match the same company across all the other datasets used in the project.

We drop the firms without an ISIN, because they cannot be linked properly with the return, market value, revenue, and carbon emission files. We also display the shape of the dataset and the available regions as a quick check before continuing with the regional selection.


In [ ]:
static_data = pd.read_excel(data_path + "Static_2025.xlsx")
static_data = static_data.dropna(subset=["ISIN"]).reset_index(drop=True)

print("Static file shape :", static_data.shape)
print("Regions available :", static_data["Region"].unique())
static_data.head()

### 1.2 Helper function for Datastream files
The Datastream files are not directly in a convenient format for the analysis. These functions clean the files and reshape them so that dates are in rows and firms are in columns, using the ISIN as the firm identifier.

We use a separate function for monthly and annual data. Monthly files already contain monthly dates, while annual files are converted to December 31 of each year because the portfolio is rebalanced at year-end. This common format makes it easier to compute returns, build the investment universe, and match financial data with carbon data later in the project.

In [ ]:
def load_monthly_datastream(filename, sheet):
    df = pd.read_excel(data_path + filename, sheet_name=sheet)
    df = df.iloc[1:].dropna(subset=["ISIN"]) #remove first row-->Datastream includes 1 extra non-company row before the actual firm observations
    df = df.drop(columns=["NAME"]).set_index("ISIN").T
    df.index = pd.to_datetime(df.index)
    df = df.apply(pd.to_numeric, errors="coerce")
    return df

def load_annual_datastream(filename, sheet):
    df = pd.read_excel(data_path + filename, sheet_name=sheet)
    df = df.iloc[1:].dropna(subset=["ISIN"])
    df = df.drop(columns=["NAME"]).set_index("ISIN").T
    df.index = pd.to_datetime([f"{int(y)}-12-31" for y in df.index])
    df = df.apply(pd.to_numeric, errors="coerce")
    return df

### 1.3 Load monthly, annual and risk-free data
Load all the datasets needed for the project. Monthly total return indices and monthly market capitalizations are used to compute portfolio returns and benchmark weights. Annual total return indices, market capitalizations, revenues, and Scope 1 CO₂ emissions are used for the yearly investment universe and carbon measures.

Load the monthly risk-free rate, which will be used later to compute excess returns and performance measures such as the Sharpe ratio.

In [ ]:
#monthly data
ri_monthly = load_monthly_datastream("DS_RI_T_USD_M_2025.xlsx", "RI")
market_cap_m = load_monthly_datastream("DS_MV_T_USD_M_2025.xlsx", "MV")

#annual data
ri_annual = load_annual_datastream("DS_RI_T_USD_Y_2025.xlsx", "RI")
market_cap_a = load_annual_datastream("DS_MV_T_USD_Y_2025.xlsx", "MV")
revenue = load_annual_datastream("DS_REV_Y_2025.xlsx", "REV")
co2_scope1_annual= load_annual_datastream("DS_CO2_SCOPE_1_Y_2025.xlsx", "Scope1")

print(ri_monthly.shape, market_cap_m.shape)
print(ri_annual.shape, market_cap_a.shape)
print(revenue.shape, co2_scope1_annual.shape)

rf = pd.read_excel(data_path + "Risk_Free_Rate_2025.xlsx",
                   sheet_name="F-F_Research_Data_Factors")
rf.columns = ["date", "rf"]
rf["date"] = pd.to_datetime(rf["date"], format="%Y%m")
rf = rf.set_index("date")
rf["rf"] = rf["rf"] / 100

### 1.4 Keep usable firms and clean RI prices
Here, we keep only firms that have usable financial data. In the raw monthly files, 2,543 firms have at least one RI observation and 2,543 firms have at least one market capitalization observation. After matching with the static file using the ISIN, we keep 2,541 firms.

All datasets are then reindexed on this same list of firms, so that prices, market values, revenues, and CO₂ emissions are aligned before the portfolio construction.

We also remove RI values below 0.5, as suggested in the project instructions, because very low RI values can create unrealistic returns. This filter removes 26,935 monthly observations.

In [ ]:
has_ri = ri_monthly.notna().any()
has_mv = market_cap_m.notna().any()
valid = has_ri & has_mv

valid_isins  = valid[valid].index
common_isins = sorted(set(static_data["ISIN"]) & set(valid_isins))

print(f"ISINs with RI data : {has_ri.sum()}")
print(f"ISINs with MV data : {has_mv.sum()}")
print(f"Numbers of firmes kept after intersection with static : {len(common_isins)}")

static_data = static_data[static_data["ISIN"].isin(common_isins)].reset_index(drop=True)
ri_monthly = ri_monthly.reindex(columns=common_isins)
market_cap_m = market_cap_m.reindex(columns=common_isins)
ri_annual = ri_annual.reindex(columns=common_isins)
market_cap_a = market_cap_a.reindex(columns=common_isins)
revenue = revenue.reindex(columns=common_isins)
co2_scope1_annual = co2_scope1_annual.reindex(columns=common_isins)

n_before = ri_monthly.notna().sum().sum()

ri_monthly = ri_monthly.where(ri_monthly >= minimum_price)
ri_annual = ri_annual.where(ri_annual >= minimum_price)

n_after = ri_monthly.notna().sum().sum()
print(f"RI values dropped (< {minimum_price}) : {n_before - n_after:,}")

### 1.5 Fill short gaps and compute monthly returns
Here, we clean the monthly RI series before calculating returns. We fill only short missing gaps, with a maximum of 2 months, to correct small data issues without keeping firms artificially alive for too long.

Apply a -100% return when a firm disappears before the end of the sample, as suggested in the project for delisted firms. This adds 201 delisting returns.


In [ ]:
first_obs = ri_monthly.apply(lambda s: s.first_valid_index())
last_obs  = ri_monthly.apply(lambda s: s.last_valid_index())
ri_filled = ri_monthly.ffill(limit=2) #fill only short gaps

for isin in ri_filled.columns:
    if pd.isna(first_obs[isin]):
        continue
    ri_filled.loc[ri_filled.index < first_obs[isin], isin] = np.nan

print(f"gaps filled: {ri_filled.notna().sum().sum() - ri_monthly.notna().sum().sum():,}")

returns = ri_filled.pct_change(fill_method=None).iloc[1:]
last_ret_date = returns.index[returns.index.year <= last_perf_year][-1]

for isin in returns.columns:
    last_date = last_obs[isin]
    if pd.isna(last_date) or last_date >= last_ret_date:
        continue
    after = returns.index[(returns.index > last_date) & (returns.index <= last_ret_date)]
    if len(after) == 0:
        continue
    returns.loc[after[0], isin] = -1.0
    if len(after) > 1:
        returns.loc[after[1:], isin] = np.nan

returns = returns.loc[:last_ret_date]
monthly_returns = returns

print(f"returns sample: {returns.index[0].date()} to {returns.index[-1].date()}")
print(f"delistings applied: {(returns == -1.0).sum().sum()}")

end_2013 = returns.index[returns.index.year == first_allocation_year][-1]
end_pos = returns.index.get_loc(end_2013)
win = returns.index[end_pos - estimation_window_months + 1 : end_pos + 1]
print(f"first estimation window: {len(win)} months")

### 1.6 Fill annual data and define stale-price filter
We fill missing annual revenue and CO₂ values using the previous available year, but only after the first real observation. This follows the project rule and avoids creating data before a firm actually starts reporting.

After this step, 2,524 firms have revenue data and 2,392 firms have Scope 1 CO₂ data. We also define a stale-price filter to remove firms with too many zero monthly returns, since they may look artificially low-risk in the minimum-variance portfolio.


In [ ]:
def ffill_after_first(df):
    first_obs = df.apply(lambda s: s.first_valid_index())
    df = df.ffill()
    for isin in df.columns:
        if pd.isna(first_obs[isin]):
            df[isin] = np.nan
        else:
            df.loc[df.index < first_obs[isin], isin] = np.nan
    return df

revenue = ffill_after_first(revenue)
co2_scope1_annual = ffill_after_first(co2_scope1_annual)

print(f"revenue firms with data: {revenue.notna().any().sum()}")
print(f"co2 firms with data: {co2_scope1_annual.notna().any().sum()}")

def find_stale_firms(returns_data, end_date, lookback_months=estimation_window_months, max_zero_share=maximum_zero_return_share): #identify firms with too many zero returns
    start = end_date - pd.DateOffset(months=lookback_months - 1)
    window = returns_data.loc[start:end_date]
    n_obs  = window.notna().sum()
    n_zero = (window == 0).sum()

    zero_share = n_zero / n_obs.replace(0, np.nan)
    stale = zero_share[zero_share > max_zero_share].index.tolist()

    return stale

## 2. Investment Setup of Groupe AI

### 2.1 Region filtrer : North America + Europe
We first restrict the universe to the regions assigned to our group. Group AI has to work on North America and Europe with Scope 1 emissions, so we keep only firms with regions `AMER` and `EUR`.

In [ ]:
regional_isins = static_data.loc[
    static_data["Region"].isin(target_regions), "ISIN"
].tolist()

for region in target_regions:
    print(f"  {region} : {(static_data['Region'] == region).sum()} firms")
print(f"  Total : {len(regional_isins)} firms in our assigned universe")

### 2.2 Yearly investment set (2013-2024)
For each allocation year, we build the list of firms that are actually investable. A firm must be in our assigned region, have Scope 1 CO₂ data, have enough return observations over the past 10 years, not be flagged as stale, and have an available RI price at the end of the year.

This gives us a different investment set each year. The universe starts with 836 firms in 2013 and increases to 1,144 firms in 2024, mainly because more firms have usable carbon and financial data over time.

In [ ]:
def build_investment_set(year, regional_isins, returns_data, carbon_data,
                         ri_data, lookback_months=estimation_window_months,
                         minimum_months=minimum_valid_months,
                         max_zero_share=maximum_zero_return_share):

    year_months = returns_data.index[returns_data.index.year == year]
    if len(year_months) == 0:
        raise ValueError(f"No monthly data for year {year}")

    end_date = year_months[-1]
    end_pos  = returns_data.index.get_loc(end_date)

    start_pos = end_pos - lookback_months + 1
    if start_pos < 0:
        raise ValueError(f"Not enough data before {year}")

    window = returns_data.iloc[start_pos:end_pos + 1]
    candidates = [i for i in regional_isins if i in returns_data.columns]

    carbon_row = carbon_data.loc[carbon_data.index.year == year]
    if len(carbon_row) == 0:
        raise ValueError(f"No carbon data for year {year}")
    has_co2 = set(carbon_row.iloc[0].dropna().index)

    n_valid    = window[candidates].notna().sum()
    enough_obs = set(n_valid[n_valid >= minimum_months].index)

    stale = set(find_stale_firms(returns_data, end_date,
                                 lookback_months, max_zero_share))

    has_price = set(ri_data.loc[end_date].dropna().index)

    inv_set = sorted(
        (set(candidates) & has_co2 & enough_obs & has_price) - stale
    )

    return inv_set


investment_set_by_year = {}
allocation_years = list(range(first_allocation_year, last_allocation_year + 1))

for year in allocation_years:
    current_set = build_investment_set(
        year=year,
        regional_isins=regional_isins,
        returns_data=monthly_returns,
        carbon_data=co2_scope1_annual,
        ri_data=ri_filled,
        lookback_months=estimation_window_months,
        minimum_months=minimum_valid_months,
        max_zero_share=maximum_zero_return_share,
    )
    investment_set_by_year[year] = current_set
    print(f"Year {year}: {len(current_set)} firms")

## 3. Estimation of Return Moments (rolling 10-year window)
For each year, we estimate expected returns and the covariance matrix using the previous 10 years of monthly returns. These are the inputs needed for the minimum-variance optimization.

We also make the covariance matrix positive semi-definite before using it in the optimizer. This avoids numerical issues when the matrix has small negative eigenvalues due to missing data or estimation noise.

In [ ]:
def estimate_moments(returns_data, investment_set, end_date,
                     lookback_months=estimation_window_months):

    end_pos   = returns_data.index.get_loc(end_date)
    start_pos = end_pos - lookback_months + 1

    window = returns_data.iloc[start_pos:end_pos + 1][investment_set]

    mu = window.mean()

    T     = len(window)
    sigma = window.cov() * (T - 1) / T
    sigma = sigma.fillna(0.0)
    s = sigma.values
    s = (s + s.T) / 2
    eigvals, eigvecs = np.linalg.eigh(s)
    s = eigvecs @ np.diag(np.maximum(eigvals, 0)) @ eigvecs.T
    s += np.eye(len(eigvals)) * 1e-7

    cov_psd = pd.DataFrame(s, index=sigma.index, columns=sigma.columns)

    return mu, cov_psd

estimated_moments = {}

for year in allocation_years:
    inv_set  = investment_set_by_year[year]
    end_date = monthly_returns.index[monthly_returns.index.year == year][-1]

    mu_year, sigma_year = estimate_moments(monthly_returns, inv_set, end_date)
    estimated_moments[year] = (mu_year, sigma_year)
    print(f"{year}: {len(inv_set)} firms, sigma {sigma_year.shape}")

# PART I — STANDARD PORTFOLIO ALLOCATION

## 4. Minimum-variance portfolio

### 4.1 Optimization and yearly weights
For each year, we solve the long-only minimum-variance portfolio problem. The goal is to find portfolio weights that minimize portfolio risk, using the covariance matrix estimated from the previous 10 years of monthly returns.

The constraints are simple: all weights must sum to 1, and short-selling is not allowed. We store the optimal weights for each year.

The largest weight shows which firm is used the most by the optimizer to lower portfolio risk. This can happen when the firm has relatively stable returns or helps diversify the rest of the portfolio.

In [ ]:
def optimize_min_variance(covariance_matrix):
    n = covariance_matrix.shape[0]
    sigma = covariance_matrix.values


    w = cp.Variable(n)

    objective = cp.Minimize(cp.quad_form(w, sigma, assume_PSD=True))
    constraints = [
        cp.sum(w) == 1,
        w >= 0]

    problem = cp.Problem(objective, constraints)
    problem.solve(solver=cp.OSQP, eps_abs=1e-8, eps_rel=1e-8, verbose=False)

    if problem.status not in ["optimal", "optimal_inaccurate"]:
        raise ValueError(f"Optimization failed: {problem.status}")

    weights = np.maximum(w.value, 0)
    weights = weights / weights.sum()

    return weights

isin_to_name = static_data.set_index("ISIN")["NAME"].to_dict()
mv_weights_by_year = {}

for year in allocation_years:
    current_investment_set = investment_set_by_year[year]
    _, covariance_matrix_year = estimated_moments[year]

    optimal_weights = optimize_min_variance(covariance_matrix_year)

    mv_weights_by_year[year] = pd.Series(
        optimal_weights,
        index=current_investment_set
    )

    n_firms_invested = (mv_weights_by_year[year] > 1e-4).sum()
    largest_isin = mv_weights_by_year[year].idxmax()
    largest_weight = mv_weights_by_year[year].max()
    largest_name = isin_to_name.get(largest_isin, "Unknown firm")

    print(
        f"In year {year}: {len(current_investment_set)} firms |"
        f"{n_firms_invested} firms invested -> largest weight = {largest_weight:.2%} in {largest_name}")

## 5. Out-of-Sample Portfolio Returns

### 5.1 Minimum-variance portfolio returns
We now use the yearly minimum-variance weights to compute the monthly out-of-sample portfolio returns. The weights are fixed at the beginning of each year, then updated month by month as stock prices move.

The portfolio return series covers 144 months, from January 2014 to December 2025.

In [ ]:

def compute_portfolio_returns(returns_data, weights_by_year, allocation_years):

    port_returns = {}

    for year in allocation_years:
        next_year_dates = returns_data.index[returns_data.index.year == year + 1]
        if len(next_year_dates) == 0:
            continue

        current_weights = weights_by_year[year].reindex(returns_data.columns).fillna(0.0)

        for date in next_year_dates:
            r_t = returns_data.loc[date]

            r_p = (current_weights * r_t).sum()
            port_returns[date] = r_p

            updated = current_weights * (1.0 + r_t.fillna(0.0))
            total = updated.sum()
            if total > 1e-6:
                current_weights = updated / total
            else:
                current_weights = current_weights * 0.0

    return pd.Series(port_returns)

returns_mv = compute_portfolio_returns(monthly_returns, mv_weights_by_year, allocation_years)

print(f"Period: {returns_mv.index[0].date()} to {returns_mv.index[-1].date()}")
print(f"Months: {len(returns_mv)}")
print(f"Monthly mean: {returns_mv.mean():.4%}")
print(f"Monthly volatility: {returns_mv.std():.4%}")
print(f"Min return: {returns_mv.min():.4%}")
print(f"Max return: {returns_mv.max():.4%}")


### 5.2 Value-weighted benchmark returns

In [ ]:
def compute_vw_returns(returns_data, market_cap_data, investment_set_by_year, allocation_years):

    benchmark_return_list = []

    for year in allocation_years:
        next_year_dates = returns_data.index[returns_data.index.year == year + 1]
        if len(next_year_dates) == 0:
            continue

        current_investment_set = investment_set_by_year[year]

        for date in next_year_dates:
            current_position = market_cap_data.index.get_loc(date)
            if current_position == 0:
                continue

            previous_date = market_cap_data.index[current_position - 1]
            previous_month_caps = market_cap_data.loc[previous_date, current_investment_set].dropna()

            if previous_month_caps.sum() <= 0:
                continue

            vw_weights = previous_month_caps / previous_month_caps.sum()

            stock_returns = returns_data.loc[date, previous_month_caps.index]
            benchmark_return = (vw_weights * stock_returns).sum()

            benchmark_return_list.append({"date": date, "return": benchmark_return})

    vw_returns_series = pd.DataFrame(benchmark_return_list).set_index("date")["return"]
    return vw_returns_series

returns_vw = compute_vw_returns(
    monthly_returns,
    market_cap_m,
    investment_set_by_year,
    allocation_years)

print(f"Period: {returns_vw.index[0].date()} to {returns_vw.index[-1].date()}")
print(f"Months: {len(returns_vw)}")
print(f"Monthly mean: {returns_vw.mean():.4%}")
print(f"Monthly volatility: {returns_vw.std():.4%}")
print(f"Min return: {returns_vw.min():.4%}")
print(f"Max return: {returns_vw.max():.4%}")

## 6. Part I Results

### 6.1 Performance statistics
We summarize the financial performance of the minimum-variance portfolio and the value-weighted benchmark. The statistics are annualized to make the comparison easier: annual return, annual volatility, Sharpe ratio, and the best/worst monthly return.

We align the risk-free rate by month before calculating the Sharpe ratio, so the comparison uses the correct excess returns instead of assuming a zero risk-free rate.

In [ ]:
def compute_performance_stats(returns_series, rf_series, label=""):
    # annualize: mu*12, vol*sqrt(12), SR = (mu - rf) / vol

    rf_aligned = rf_series.reindex(returns_series.index).fillna(0.0)

    mu  = returns_series.mean() * 12
    vol = returns_series.std()  * np.sqrt(12)
    rf  = rf_aligned.mean()     * 12
    sr  = (mu - rf) / vol

    return {
        "Portfolio"      : label,
        "Ann. Return"    : f"{mu:.2%}",
        "Ann. Volatility": f"{vol:.2%}",
        "Sharpe Ratio"   : f"{sr:.3f}",
        "Min (monthly)"  : f"{returns_series.min():.2%}",
        "Max (monthly)"  : f"{returns_series.max():.2%}",}

stats_mv = compute_performance_stats(returns_mv, rf["rf"], "Minimum-Variance Portfolio")
stats_vw = compute_performance_stats(returns_vw, rf["rf"], "Value-Weighted Portfolio")

stats_df = pd.DataFrame([stats_mv, stats_vw]).set_index("Portfolio")
print(stats_df.to_string())

### 6.2 Cumulative returns plot
We plot the cumulative performance of the minimum-variance portfolio and the value-weighted benchmark. Both series start from the same initial value, so the graph shows how the two strategies evolve over time.

This plot is mainly used to compare the realized out-of-sample performance from 2014 to 2025.

In [ ]:
fig, axis = plt.subplots(figsize=(12, 6))

wealth_mv = portfolio_value * (1 + returns_mv).cumprod()
wealth_vw = portfolio_value * (1 + returns_vw).cumprod()

axis.plot(wealth_mv.index, wealth_mv.values,
    label="Minimum-Variance Portfolio",
    color="blue",
    linewidth=1.8
)

axis.plot(wealth_vw.index, wealth_vw.values,
    label="Value-Weighted Portfolio",
    color="red",
    linewidth=1.8
)

axis.axhline(y=portfolio_value, color="black", linestyle="--", linewidth=0.8, alpha=0.5)

axis.set_title("Portfolio Value – Minimum-Variance vs Value-Weighted\n"
    "(North America + Europe, Scope 1, 2014–2025)",
    fontsize=13)

axis.set_xlabel("Date")
axis.set_ylabel("Portfolio value (million USD)")
axis.legend(fontsize=11)
axis.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
axis.xaxis.set_major_locator(mdates.YearLocator())

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# PART II — PORTFOLIO ALLOCATION WITH CARBON EMISSION REDUCTION

## 7. Carbon Metrics

### 7.1 Carbon intensity (CI)
We compute carbon intensity as Scope 1 CO₂ emissions divided by revenues. Since revenues are given in thousands of USD, we divide them by 1,000 to express revenues in million USD. The final unit is therefore tonnes of CO₂ per million USD of revenue.

For 2013, carbon intensity can be computed for 1,390 firms. The distribution is very skewed: the median is 12.7 tCO₂/MUSD, while the mean is 324.7 tCO₂/MUSD. This means that a few very carbon-intensive firms strongly increase the average.

In [ ]:
carbon_intensity = co2_scope1_annual / (revenue / 1000)

ci_2013 = carbon_intensity.loc[carbon_intensity.index.year == 2013].iloc[0].dropna()

print(f"Carbon intensity available for {len(ci_2013)} firms in 2013")
print(f"Median: {ci_2013.median():.1f} tCO2/MUSD")
print(f"Mean: {ci_2013.mean():.1f} tCO2/MUSD")
print(f"P95: {ci_2013.quantile(0.95):.1f} tCO2/MUSD")
print(f"Max: {ci_2013.max():.1f} tCO2/MUSD")

### 7.2 Portfolio carbon metrics: WACI and Carbon Footprint
Compute two carbon metrics for the portfolios. WACI measures the portfolio’s exposure to carbon-intensive firms, using portfolio weights and firm carbon intensity. Carbon footprint measures the emissions attributed to the investor per million USD invested.  
We first test the carbon metric functions on 2013 because it is the first allocation year. The portfolio built at the end of 2013 is used for the 2014 investment period. Later, the same metrics will be computed for all allocation years.


In [ ]:
def compute_waci(portfolio_weights, carbon_intensity_year):
    common_isins = portfolio_weights.index.intersection(
        carbon_intensity_year.dropna().index
    )

    weights = portfolio_weights[common_isins]
    weights = weights / weights.sum()

    return (weights * carbon_intensity_year[common_isins]).sum()


def compute_carbon_footprint(portfolio_weights, emissions_year,
                             market_cap_year, portfolio_value):
    common_isins = portfolio_weights.index
    common_isins = common_isins.intersection(emissions_year.dropna().index)
    common_isins = common_isins.intersection(market_cap_year.dropna().index)

    ownership_share = (
        portfolio_weights[common_isins] * portfolio_value / market_cap_year[common_isins]
    )

    carbon_footprint = (
        ownership_share * emissions_year[common_isins]
    ).sum() / portfolio_value

    return carbon_footprint


def compute_benchmark_carbon_footprint(emissions_year, market_cap_year):
    common_isins = emissions_year.dropna().index.intersection(
        market_cap_year.dropna().index
    )

    total_market_cap = market_cap_year[common_isins].sum()
    total_emissions = emissions_year[common_isins].sum()

    return total_emissions / total_market_cap


investment_set_2013 = investment_set_by_year[2013]
carbon_intensity_2013 = carbon_intensity.loc[carbon_intensity.index.year == 2013].iloc[0]
emissions_2013 = co2_scope1_annual.loc[co2_scope1_annual.index.year == 2013].iloc[0]
market_cap_2013 = market_cap_a.loc[market_cap_a.index.year == 2013].iloc[0]

waci_2013 = compute_waci(mv_weights_by_year[2013], carbon_intensity_2013)

carbon_footprint_2013 = compute_carbon_footprint(
    mv_weights_by_year[2013],
    emissions_2013,
    market_cap_2013,
    portfolio_value
)

benchmark_carbon_footprint_2013 = compute_benchmark_carbon_footprint(
    emissions_2013[investment_set_2013],
    market_cap_2013[investment_set_2013]
)

print("Year 2013:")
print(f"WACI (Minimum-Variance Portfolio): {waci_2013:.1f} tCO2/MUSD revenue")
print(f"Carbon Footprint (Minimum-Variance Portfolio): {carbon_footprint_2013:.1f} tCO2/MUSD invested")
print(f"Carbon Footprint (Value-Weighted Portfolio): {benchmark_carbon_footprint_2013:.1f} tCO2/MUSD invested")

## 8. Carbon Profile of Existing Portfolios

### 8.1 Standard portfolios: wealth, WACI and carbon footprint
We now compute WACI and carbon footprint for the minimum-variance portfolio and the value-weighted benchmark for every allocation year from 2013 to 2024.

For each year, we use the portfolio weights, Scope 1 emissions, carbon intensity, and market capitalization available at year-end.

In [ ]:
# Yearly WACI and carbon footprint for the two standard portfolios
waci_mv_by_year = {}
waci_vw_by_year = {}
cf_mv_by_year = {}
cf_vw_by_year = {}

wealth_mv = portfolio_value
wealth_vw = portfolio_value

for year in allocation_years:

    ci_year = carbon_intensity.loc[carbon_intensity.index.year == year].iloc[0]
    co2_year = co2_scope1_annual.loc[co2_scope1_annual.index.year == year].iloc[0]
    cap_year = market_cap_a.loc[market_cap_a.index.year == year].iloc[0]

    inv_set = investment_set_by_year[year]
    mv_weights = mv_weights_by_year[year]

    waci_mv_by_year[year] = compute_waci(mv_weights, ci_year)
    cf_mv_by_year[year] = compute_carbon_footprint(
        mv_weights, co2_year, cap_year, wealth_mv
    )

    caps_inv = cap_year[inv_set].dropna()
    vw_weights = caps_inv / caps_inv.sum()

    waci_vw_by_year[year] = compute_waci(vw_weights, ci_year)
    cf_vw_by_year[year] = compute_benchmark_carbon_footprint(
        co2_year[inv_set], cap_year[inv_set]
    )
    next_year_returns_mv = returns_mv[returns_mv.index.year == year + 1]
    next_year_returns_vw = returns_vw[returns_vw.index.year == year + 1]

    if len(next_year_returns_mv) > 0:
        wealth_mv *= (1 + next_year_returns_mv).prod()

    if len(next_year_returns_vw) > 0:
        wealth_vw *= (1 + next_year_returns_vw).prod()

    print(
        f"Year {year} | "
        f"WACI MV = {waci_mv_by_year[year]:.1f} | "
        f"CF MV = {cf_mv_by_year[year]:.1f} | "
        f"WACI VW = {waci_vw_by_year[year]:.1f} | "
        f"CF VW = {cf_vw_by_year[year]:.1f}"
    )

### 8.2 Firms driving carbon intensity
Le projet demande explicitement d’identifier les firmes qui font monter l’intensité carbone, par exemple le top 10 avec les noms et ISIN.

In [ ]:
top_n = 10

company_names = static_data.set_index("ISIN")["NAME"]

print("=" * 75)
print(f"Top {top_n} firms driving WACI in the minimum-variance portfolio")
print("=" * 75)

for year in [first_allocation_year, last_allocation_year]:

    ci_year = carbon_intensity.loc[carbon_intensity.index.year == year].iloc[0]
    mv_weights = mv_weights_by_year[year]

    common = mv_weights.index.intersection(ci_year.dropna().index)
    contribution = (mv_weights[common] * ci_year[common]).sort_values(ascending=False)

    total_waci = contribution.sum()

    print(f"\nYear {year} (WACI = {total_waci:.1f} tCO2/MUSD)")
    print(f"{'Rank':<6}{'ISIN':<15}{'Name':<40}{'Weight':>8}{'CI':>10}{'Contribution':>14}")
    print("-" * 95)

    for rank, (isin, contrib) in enumerate(contribution.head(top_n).items(), 1):
        name = company_names.get(isin, "N/A")[:38]
        weight = mv_weights[isin]
        ci_val = ci_year[isin]

        print(f"{rank:<6}{isin:<15}{name:<40}{weight:>7.2%}{ci_val:>10.1f}{contrib:>14.1f}")

    top_share = contribution.head(top_n).sum() / total_waci
    print(f"\nTop {top_n} firms account for {top_share:.1%} of total WACI")

### 8.3 Carbon metrics over time
Plot the yearly WACI and carbon footprint of the minimum-variance portfolio and the value-weighted benchmark. This helps us see how the carbon exposure of both portfolios changes over the full allocation period.

The goal is to compare the standard minimum-variance portfolio with the passive benchmark before adding any carbon constraint.

In [ ]:
years_list = sorted(waci_mv_by_year.keys())

waci_mv_series = [waci_mv_by_year[y] for y in years_list]
waci_vw_series = [waci_vw_by_year[y] for y in years_list]
cf_mv_series = [cf_mv_by_year[y] for y in years_list]
cf_vw_series = [cf_vw_by_year[y] for y in years_list]

fig, axis = plt.subplots(1, 2, figsize=(14, 5))

#WACI
axis[0].plot(
    years_list,
    waci_mv_series,
    marker="o",
    markersize=4,
    label="Minimum-Variance Portfolio",
    color="blue",
    linewidth=1.8
)

axis[0].plot(
    years_list,
    waci_vw_series,
    marker="s",
    markersize=4,
    label="Value-Weighted Portfolio",
    color="red",
    linewidth=1.8
)

axis[0].set_title("WACI over Time")
axis[0].set_xlabel("Year")
axis[0].set_ylabel("WACI (tCO₂ / MUSD revenue)")
axis[0].legend()
axis[0].set_xticks(years_list)
axis[0].tick_params(axis="x", rotation=45)

#carbon footprint
axis[1].plot(
    years_list,
    cf_mv_series,
    marker="o",
    markersize=4,
    label="Minimum-Variance Portfolio",
    color="blue",
    linewidth=1.8
)

axis[1].plot(
    years_list,
    cf_vw_series,
    marker="s",
    markersize=4,
    label="Value-Weighted Portfolio",
    color="red",
    linewidth=1.8
)

axis[1].set_title("Carbon Footprint over Time")
axis[1].set_xlabel("Year")
axis[1].set_ylabel("Carbon Footprint (tCO₂ / MUSD invested)")
axis[1].legend()
axis[1].set_xticks(years_list)
axis[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

print(f"WACI change from {years_list[0]} to {years_list[-1]}:")
print(f"Minimum-Variance Portfolio: {waci_mv_series[0]:.1f} -> {waci_mv_series[-1]:.1f}"
    f"({(waci_mv_series[-1] / waci_mv_series[0] - 1) * 100:+.1f}%)"
)
print(f"Value-Weighted Portfolio: {waci_vw_series[0]:.1f} -> {waci_vw_series[-1]:.1f}"
    f"({(waci_vw_series[-1] / waci_vw_series[0] - 1) * 100:+.1f}%)"
)

print(f"\nCarbon footprint change from {years_list[0]} to {years_list[-1]}:")
print(f"Minimum-Variance Portfolio: {cf_mv_series[0]:.1f} -> {cf_mv_series[-1]:.1f}"
    f"({(cf_mv_series[-1] / cf_mv_series[0] - 1) * 100:+.1f}%)"
)
print(f"Value-Weighted Portfolio: {cf_vw_series[0]:.1f} -> {cf_vw_series[-1]:.1f} "
    f"({(cf_vw_series[-1] / cf_vw_series[0] - 1) * 100:+.1f}%)"
)

To understand the carbon spikes of the minimum-variance portfolio, we look at the firms contributing the most to WACI in 2014 and 2018. These two years are selected because they show the largest increases in the carbon metrics.

A firm contributes strongly to WACI when it has a high portfolio weight, a high carbon intensity, or both. This check helps identify whether the carbon exposure is spread across many firms or mainly driven by a few names.

In [ ]:
company_names = static_data.set_index("ISIN")["NAME"]
years_to_check = [2014, 2018]
top_n = 10

for year in years_to_check:
    ci_year = carbon_intensity.loc[carbon_intensity.index.year == year].iloc[0]
    mv_weights = mv_weights_by_year[year]

    common = mv_weights.index.intersection(ci_year.dropna().index)
    contribution = (mv_weights[common] * ci_year[common]).sort_values(ascending=False)

    total_waci = contribution.sum()

    rows = []

    for rank, (isin, contrib) in enumerate(contribution.head(top_n).items(), 1):
        rows.append({
            "Rank": rank,
            "ISIN": isin,
            "Name": company_names.get(isin, "N/A"),
            "Weight": mv_weights[isin],
            "CI": ci_year[isin],
            "Contribution": contrib,
            "Share of WACI": contrib / total_waci
        })

    top_table = pd.DataFrame(rows)

    print(f"\nYear {year} - Top {top_n} WACI contributors")
    print(f"Total WACI: {total_waci:.1f} tCO2/MUSD")
    print(f"Top 1 share: {contribution.head(1).sum() / total_waci:.1%}")
    print(f"Top {top_n} share: {contribution.head(top_n).sum() / total_waci:.1%}")

    display(top_table)

## 9. Minimum-Variance Portfolio with 50% Carbon Reduction

### 9.1 Optimization under the 50% carbon footprint constraint
### 9.1 Optimization under the carbon footprint constraint

We add a carbon footprint constraint to the minimum-variance portfolio. The objective is still to minimize portfolio variance.

The threshold is defined at the beginning of the notebook with `carbon_reduction_target`, so it can easily be changed to test a different reduction level.

In [ ]:
def optimize_min_variance_with_cf_cap(covariance_matrix, emissions_year, market_cap_year, cf_cap):
    n = covariance_matrix.shape[0]
    sigma = covariance_matrix.values
    isins = list(covariance_matrix.index)

    c_vector = np.zeros(n)

    for i, isin in enumerate(isins):
        emissions = emissions_year.get(isin, np.nan)
        market_cap = market_cap_year.get(isin, np.nan)

        if pd.notna(emissions) and pd.notna(market_cap) and market_cap > 0:
            c_vector[i] = emissions / market_cap
        else:
            raise ValueError(f"missing carbon or market cap data for {isin}")

    weights = cp.Variable(n)

    objective = cp.Minimize(cp.quad_form(weights, sigma, assume_PSD=True))
    constraints = [
        cp.sum(weights) == 1,
        weights >= 0,
        c_vector @ weights <= cf_cap]

    problem = cp.Problem(objective, constraints)
    problem.solve(solver=cp.OSQP, eps_abs=1e-8, eps_rel=1e-8, verbose=False)

    optimal_weights = np.maximum(weights.value, 0)
    optimal_weights = optimal_weights / optimal_weights.sum()

    return optimal_weights

mv_50_weights_by_year = {}

for year in allocation_years:
    current_investment_set = investment_set_by_year[year]
    _, covariance_matrix_year = estimated_moments[year]

    emissions_year = co2_scope1_annual.loc[co2_scope1_annual.index.year == year].iloc[0]

    market_cap_year = market_cap_a.loc[
        market_cap_a.index.year == year
    ].iloc[0]

    cf_cap = carbon_reduction_target * cf_mv_by_year[year]

    optimal_weights = optimize_min_variance_with_cf_cap(
        covariance_matrix_year,
        emissions_year,
        market_cap_year,
        cf_cap
    )
    mv_50_weights_by_year[year] = pd.Series(
        optimal_weights,
        index=current_investment_set)

    realized_cf = compute_carbon_footprint(
        mv_50_weights_by_year[year],
        emissions_year,
        market_cap_year,
        portfolio_value
    )

    n_firms_invested = (mv_50_weights_by_year[year] > 1e-4).sum()
    largest_isin = mv_50_weights_by_year[year].idxmax()
    largest_weight = mv_50_weights_by_year[year].max()
    largest_name = isin_to_name.get(largest_isin, "Unknown")

    print(f"In year {year}: {len(current_investment_set)} firms | "
        f"{n_firms_invested} firms invested -> largest weight = {largest_weight:.2%} in {largest_name}| "
        f"CF = {realized_cf:.1f} vs target = {cf_cap:.1f}"
    )

### 9.2 Out-of-sample returns

In [ ]:
returns_mv_50 = compute_portfolio_returns(
    monthly_returns,
    mv_50_weights_by_year,
    allocation_years)

print(f"Period: {returns_mv_50.index[0].date()} to {returns_mv_50.index[-1].date()}")
print(f"Months: {len(returns_mv_50)}")
print(f"Monthly mean: {returns_mv_50.mean():.4%}")
print(f"Monthly volatility: {returns_mv_50.std():.4%}")
print(f"Min return: {returns_mv_50.min():.4%}")
print(f"Max return: {returns_mv_50.max():.4%}")

### 9.3 Performance, WACI and Carbon Footprint
### 9.3 Comparison with the 50% carbon-constrained portfolio

We compare the standard minimum-variance portfolio with the minimum-variance portfolio under the carbon footprint constraint. We report both the financial performance and the yearly carbon metrics.  

The graphs show whether the 50% carbon footprint constraint effectively reduces carbon exposure, and whether this reduction comes with a visible cost in terms of portfolio performance.

In [ ]:

stats_mv_50 = compute_performance_stats(
    returns_mv_50,
    rf["rf"],
    "Minimum-Variance Portfolio (50% constraint)"
)

stats_compare_mv = pd.DataFrame([stats_mv, stats_mv_50]).set_index("Portfolio")
print("Performance summary: standard vs carbon-constrained minimum-variance portfolio\n")
print(stats_compare_mv.to_string())

waci_mv_50_by_year = {}
cf_mv_50_by_year = {}

for year in allocation_years:
    ci_year = carbon_intensity.loc[carbon_intensity.index.year == year].iloc[0]
    co2_year = co2_scope1_annual.loc[co2_scope1_annual.index.year == year].iloc[0]
    cap_year = market_cap_a.loc[market_cap_a.index.year == year].iloc[0]

    weights_50 = mv_50_weights_by_year[year]

    waci_mv_50_by_year[year] = compute_waci(weights_50, ci_year)
    cf_mv_50_by_year[year] = compute_carbon_footprint(
        weights_50,
        co2_year,
        cap_year,
        portfolio_value)

    target_cap = carbon_reduction_target * cf_mv_by_year[year]

years_list = sorted(waci_mv_by_year.keys())

waci_mv_series = [waci_mv_by_year[y] for y in years_list]
waci_mv_50_series = [waci_mv_50_by_year[y] for y in years_list]

cf_mv_series = [cf_mv_by_year[y] for y in years_list]
cf_mv_50_series = [cf_mv_50_by_year[y] for y in years_list]
cf_target_series = [carbon_reduction_target * x for x in cf_mv_series]

cum_mv = (1 + returns_mv).cumprod()
cum_mv_50 = (1 + returns_mv_50).cumprod()

fig, axis = plt.subplots(1, 3, figsize=(18, 5))

axis[0].plot(
    years_list,
    waci_mv_series,
    marker="o",
    markersize=4,
    label="Minimum-Variance Portfolio",
    color="blue",
    linewidth=1.8)

axis[0].plot(years_list, waci_mv_50_series,
    marker="s",
    markersize=4,
    label="Constrained Portfolio",
    color="green",
    linewidth=1.8)

axis[0].set_title("WACI over Time")
axis[0].set_xlabel("Year")
axis[0].set_ylabel("WACI (tCO₂ / MUSD revenue)")
axis[0].legend()
axis[0].set_xticks(years_list)
axis[0].tick_params(axis="x", rotation=45)

axis[1].plot(years_list, cf_mv_series,
    marker="o",
    markersize=4,
    label="Minimum-Variance Portfolio",
    color="blue",
    linewidth=1.8)

axis[1].plot(years_list, cf_mv_50_series,
    marker="s",
    markersize=4,
    label="Constrained Portfolio",
    color="green",
    linewidth=1.8,
    zorder=2)

axis[1].plot(years_list, cf_target_series,
    marker="x",
    markersize=6,
    linestyle="--",
    color="black",
    linewidth=1.4,
    label=f"{int(carbon_reduction_target * 100)}% target",
    zorder=3)
axis[1].set_title("Carbon Footprint over Time")
axis[1].set_xlabel("Year")
axis[1].set_ylabel("Carbon Footprint (tCO₂ / MUSD invested)")
axis[1].legend()
axis[1].set_xticks(years_list)
axis[1].tick_params(axis="x", rotation=45)

axis[2].plot(
    cum_mv.index,
    cum_mv.values,
    label="Minimum-Variance Portfolio",
    color="blue",
    linewidth=1.8)

axis[2].plot(cum_mv_50.index, cum_mv_50.values,
    label="Constrained Portfolio",
    color="green",
    linewidth=1.8)

axis[2].axhline(
    y=1,
    color="black",
    linestyle="--",
    linewidth=0.8,
    alpha=0.5)

axis[2].set_title("Cumulative Performance")
axis[2].set_xlabel("Date")
axis[2].set_ylabel("Cumulative performance (base 1)")
axis[2].legend()
axis[2].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
axis[2].xaxis.set_major_locator(mdates.YearLocator())
axis[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

### 9.4 Portfolio composition changes
We compare the standard minimum-variance portfolio with the carbon-constrained version. The goal is to identify which firms are reduced or increased after adding the carbon footprint constraint.  

This helps explain how the optimizer achieves the carbon reduction.

In [ ]:
company_names = static_data.set_index("ISIN")["NAME"]
top_changes_n = 5
years_to_check = [first_allocation_year, last_allocation_year]

for year in years_to_check:

    w_mv = mv_weights_by_year[year]
    w_mv_50 = mv_50_weights_by_year[year]
    co2_year = co2_scope1_annual.loc[co2_scope1_annual.index.year == year].iloc[0]
    cap_year = market_cap_a.loc[market_cap_a.index.year == year].iloc[0]

    carbon_factor = co2_year / cap_year
    all_isins = w_mv.index.union(w_mv_50.index)

    w_mv = w_mv.reindex(all_isins).fillna(0.0)
    w_mv_50 = w_mv_50.reindex(all_isins).fillna(0.0)

    delta = w_mv_50 - w_mv

    comparison = pd.DataFrame({
        "Name": [company_names.get(isin, "N/A") for isin in all_isins],
        "w_mv": w_mv,
        "w_mv_constrained": w_mv_50,
        "Change": delta,
        "CO2_per_market_cap": carbon_factor.reindex(all_isins)
    })

    comparison = comparison.sort_values("Change")

    print(f"Year {year} - Portfolio composition changes")

    print(f"\nTop {top_changes_n} most reduced firms:")
    display(
        comparison.head(top_changes_n)[
            ["Name", "w_mv", "w_mv_constrained", "Change", "CO2_per_market_cap"]
        ].style.format({
            "w_mv": "{:.2%}",
            "w_mv_constrained": "{:.2%}",
            "Change": "{:+.2%}",
            "CO2_per_market_cap": "{:.1f}"
        }))

    print(f"\nTop {top_changes_n} most increased firms:")
    display(comparison.tail(top_changes_n).sort_values("Change", ascending=False)[
            ["Name", "w_mv", "w_mv_constrained", "Change", "CO2_per_market_cap"]
        ].style.format({
            "w_mv": "{:.2%}",
            "w_mv_constrained": "{:.2%}",
            "Change": "{:+.2%}",
            "CO2_per_market_cap": "{:.1f}"
        }))

## 10. Value-Weighted Portfolio with 50% Carbon Reduction

### 10.1 Tracking-error optimization
We use OSQP directly rather than through cvxpy for this optimization.
When using cvxpy with `cp.quad_form`, we observed that the solver
systematically over-satisfied the carbon constraint, producing CF values
around 9–10 instead of the expected 25–30. The root cause is a numerical
scaling issue: the tracking-error objective (TE² ~ 1e-8) and the carbon
constraint (CF ~ 50–175) operate on very different scales, which causes
cvxpy's interface to OSQP to treat the carbon constraint as dominant and
sacrifice tracking accuracy to minimize CF far below the target.

Calling OSQP directly resolves this: OSQP handles mixed-scale problems
natively via its internal equilibration. The optimization problem is
mathematically identical to the cvxpy formulation — we simply pass
**P = 2Σ** and **q = −2Σw_vw** (the gradient of the TE² objective)
directly to the solver, bypassing cvxpy's pre-processing layer.

In [ ]:
import osqp
import scipy.sparse as sp

def optimize_tracking_error_with_cf_cap(covariance_matrix, benchmark_weights, emissions_year, market_cap_year, cf_cap):

    n = covariance_matrix.shape[0]
    isins = list(covariance_matrix.index)

    sigma_np = (covariance_matrix.values + covariance_matrix.values.T) / 2
    w_vw = benchmark_weights.reindex(isins).fillna(0.0).values

    P = sp.csc_matrix(2.0 * sigma_np)
    q = -2.0 * sigma_np @ w_vw

    c_vector = np.zeros(n)
    for i, isin in enumerate(isins):
        e = emissions_year.get(isin, np.nan)
        m = market_cap_year.get(isin, np.nan)
        if pd.notna(e) and pd.notna(m) and m > 0:
            c_vector[i] = e / m

    A = sp.vstack([
        sp.csc_matrix(np.ones((1, n))),
        sp.eye(n, format="csc"),
        sp.csc_matrix(c_vector.reshape(1, -1))
    ], format="csc")

    l = np.concatenate([[1.0], np.zeros(n), [-np.inf]])
    u = np.concatenate([[1.0], np.ones(n),  [cf_cap]])

    solver = osqp.OSQP()
    solver.setup(P, q, A, l, u,
                 eps_abs=1e-8,
                 eps_rel=1e-8,
                 verbose=False)
    result = solver.solve()

    if result.info.status not in ["solved", "solved inaccurate"]:
        print(f"  Warning: {result.info.status} — fallback to VW weights")
        return w_vw

    weights = np.maximum(result.x, 0.0)
    weights /= weights.sum()
    return pd.Series(weights, index=isins)

vw_weights_by_year = {}

for year in allocation_years:
    cap_year = market_cap_a.loc[market_cap_a.index.year == year].iloc[0]
    inv_set = investment_set_by_year[year]
    caps_inv = cap_year[inv_set].dropna()
    vw_weights_by_year[year] = caps_inv / caps_inv.sum()

vw_50_weights_by_year = {}

for year in allocation_years:

    current_investment_set = investment_set_by_year[year]
    _, covariance_matrix_year = estimated_moments[year]

    emissions_year = co2_scope1_annual.loc[
        co2_scope1_annual.index.year == year].iloc[0]
    market_cap_year = market_cap_a.loc[
        market_cap_a.index.year == year].iloc[0]

    w_vw_aligned = vw_weights_by_year[year].reindex(
        current_investment_set).fillna(0.0)

    cf_cap = carbon_reduction_target * cf_vw_by_year[year]

    optimal_weights = optimize_tracking_error_with_cf_cap(
        covariance_matrix_year,
        w_vw_aligned,
        emissions_year,
        market_cap_year,
        cf_cap
    )

    vw_50_weights_by_year[year] = pd.Series(
        optimal_weights,
        index=current_investment_set
    )

    n_firms_invested = (vw_50_weights_by_year[year] > 1e-4).sum()
    largest_isin     = vw_50_weights_by_year[year].idxmax()
    largest_weight   = vw_50_weights_by_year[year].max()
    largest_name     = isin_to_name.get(largest_isin, "Unknown")

    realized_cf = compute_carbon_footprint(
        vw_50_weights_by_year[year],
        emissions_year,
        market_cap_year,
        portfolio_value
    )

    print(f"Year {year}: {len(current_investment_set)} firms | "
        f"{n_firms_invested} firms invested -> largest weight = {largest_weight:.2%} in {largest_name} |"
        f"CF = {realized_cf:.1f} vs target = {cf_cap:.1f}"
    )

### 10.2 Out-of-sample returns

In [ ]:
returns_vw_50 = compute_portfolio_returns(
    monthly_returns,
    vw_50_weights_by_year,
    allocation_years)

print(f"Period: {returns_vw_50.index[0].date()} to {returns_vw_50.index[-1].date()}")
print(f"Months: {len(returns_vw_50)}")
print(f"Monthly mean: {returns_vw_50.mean():.4%}")
print(f"Monthly volatility: {returns_vw_50.std():.4%}")
print(f"Min return: {returns_vw_50.min():.4%}")
print(f"Max return: {returns_vw_50.max():.4%}")

### 10.3 Performance, WACI and Carbon Footprint

In [ ]:
stats_vw_50 = compute_performance_stats(
    returns_vw_50, rf["rf"], "Value-Weighted with 50% CF reduction"
)

stats_compare_vw = pd.DataFrame([stats_vw, stats_vw_50]).set_index("Portfolio")
print("Performance summary: standard vs carbon-constrained value-weighted portfolio\n")
print(stats_compare_vw.to_string())

waci_vw_50_by_year = {}
cf_vw_50_by_year   = {}

for year in allocation_years:

    ci_year  = carbon_intensity.loc[carbon_intensity.index.year == year].iloc[0]
    co2_year = co2_scope1_annual.loc[co2_scope1_annual.index.year == year].iloc[0]
    cap_year = market_cap_a.loc[market_cap_a.index.year == year].iloc[0]

    w = vw_50_weights_by_year[year]

    waci_vw_50_by_year[year] = compute_waci(w, ci_year)
    cf_vw_50_by_year[year]   = compute_carbon_footprint(
        w, co2_year, cap_year, portfolio_value)

    target_cap = carbon_reduction_target * cf_vw_by_year[year]

years_list = sorted(waci_vw_by_year.keys())

waci_vw_series = [waci_vw_by_year[y] for y in years_list]
waci_vw_50_series = [waci_vw_50_by_year[y] for y in years_list]
cf_vw_series = [cf_vw_by_year[y] for y in years_list]
cf_vw_50_series = [cf_vw_50_by_year[y] for y in years_list]
cf_target_series = [carbon_reduction_target * x for x in cf_vw_series]

cum_vw = (1 + returns_vw).cumprod()
cum_vw_50 = (1 + returns_vw_50).cumprod()

fig, axis = plt.subplots(1, 3, figsize=(18, 5))

axis[0].plot(years_list, waci_vw_series,
    marker="o", markersize=4,
    label="Value-Weighted Portfolio",
    color="red", linewidth=1.8)
axis[0].plot(years_list, waci_vw_50_series,
    marker="s", markersize=4,
    label="Constrained Portfolio",
    color="green", linewidth=1.8)

axis[0].set_title("WACI over Time")
axis[0].set_xlabel("Year")
axis[0].set_ylabel("WACI (tCO₂ / MUSD revenue)")
axis[0].legend()
axis[0].set_xticks(years_list)
axis[0].tick_params(axis="x", rotation=45)

axis[1].plot(years_list, cf_vw_series,
    marker="o", markersize=4,
    label="Value-Weighted Portfolio",
    color="red", linewidth=1.8
)
axis[1].plot(years_list, cf_vw_50_series,
    marker="s", markersize=4,
    label="Constrained Portfolio",
    color="green", linewidth=1.8,
    zorder=2)

axis[1].plot(years_list, cf_target_series,
    marker="x", markersize=6,
    linestyle="--", color="black", linewidth=1.4,
    label=f"{int(carbon_reduction_target * 100)}% target",
    zorder=3
)
axis[1].set_title("Carbon Footprint over Time")
axis[1].set_xlabel("Year")
axis[1].set_ylabel("Carbon Footprint (tCO₂ / MUSD invested)")
axis[1].legend()
axis[1].set_xticks(years_list)
axis[1].tick_params(axis="x", rotation=45)

axis[2].plot(
    cum_vw.index, cum_vw.values,
    label="Value-Weighted Portfolio",
    color="red", linewidth=1.8
)
axis[2].plot(
    cum_vw_50.index, cum_vw_50.values,
    label="Constrained Portfolio",
    color="green", linewidth=1.8
)
axis[2].axhline(
    y=1, color="black", linestyle="--", linewidth=0.8, alpha=0.5
)
axis[2].set_title("Cumulative Performance")
axis[2].set_xlabel("Date")
axis[2].set_ylabel("Cumulative performance (base 1)")
axis[2].legend()
axis[2].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
axis[2].xaxis.set_major_locator(mdates.YearLocator())
axis[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

te_annual = (returns_vw_50 - returns_vw).std() * np.sqrt(12)
print(f"\nRealised annualised tracking error: {te_annual:.2%}")

## 11. Comparison of the Four Portfolios

### 11.1 Summary table

In [ ]:
summary_df = pd.DataFrame([stats_mv, stats_mv_50, stats_vw, stats_vw_50]).set_index("Portfolio")

def cum_ret(r):
    return (1 + r).prod() - 1

summary_df["Cumul. Return"] = [f"{cum_ret(r):.2%}" for r in [returns_mv, returns_mv_50, returns_vw, returns_vw_50]]
summary_df["Avg WACI"] = [f"{np.mean(list(d.values())):.1f}" for d in [waci_mv_by_year, waci_mv_50_by_year,
                                                   waci_vw_by_year, waci_vw_50_by_year]]
summary_df["Avg CF"] = [f"{np.mean(list(d.values())):.1f}" for d in [cf_mv_by_year, cf_mv_50_by_year,
                                                   cf_vw_by_year, cf_vw_50_by_year]]

print("Performance summary: four portfolios\n")
print(summary_df.to_string())

### 11.2 Cumulative returns and carbon footprint plots

In [ ]:
# cumulative returns and carbon metrics — 4 portfolios side by side

cum_mv    = (1 + returns_mv).cumprod()
cum_mv_50 = (1 + returns_mv_50).cumprod()
cum_vw    = (1 + returns_vw).cumprod()
cum_vw_50 = (1 + returns_vw_50).cumprod()

years_list = sorted(waci_mv_by_year.keys())

fig, axis = plt.subplots(1, 3, figsize=(18, 5))

# cumulative returns
axis[0].plot(cum_mv.index, cum_mv.values, label="Min-Variance Portfolio", color="blue", linewidth=1.8)
axis[0].plot(cum_mv_50.index, cum_mv_50.values,
             label="Min-Variance (50% CF)",    color="green",       linewidth=1.8)
axis[0].plot(cum_vw.index,    cum_vw.values,
             label="Value-Weighted Portfolio", color="red",  linewidth=1.8)
axis[0].plot(cum_vw_50.index, cum_vw_50.values,
             label="Value-Weighted (50% CF)",  color="orange",      linewidth=1.8)
axis[0].axhline(y=1, color="black", linestyle="--", linewidth=0.8, alpha=0.5)
axis[0].set_title("Cumulative Performance (4 portfolios)")
axis[0].set_xlabel("Date")
axis[0].set_ylabel("Cumulative performance (base 1)")
axis[0].legend(fontsize=9)
axis[0].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
axis[0].xaxis.set_major_locator(mdates.YearLocator())
axis[0].tick_params(axis="x", rotation=45)

axis[1].plot(years_list, [waci_mv_by_year[y]    for y in years_list],
             marker="o", label="Min-Variance Portfolio",   color="blue")
axis[1].plot(years_list, [waci_mv_50_by_year[y] for y in years_list],
             marker="s", label="Min-Variance (50% CF)",    color="green")
axis[1].plot(years_list, [waci_vw_by_year[y]    for y in years_list],
             marker="o", label="Value-Weighted Portfolio", color="red")
axis[1].plot(years_list, [waci_vw_50_by_year[y] for y in years_list],
             marker="s", label="Value-Weighted (50% CF)",  color="orange")
axis[1].set_title("WACI over Time (4 portfolios)")
axis[1].set_xlabel("Year")
axis[1].set_ylabel("WACI (tCO₂ / MUSD revenue)")
axis[1].legend(fontsize=9)
axis[1].set_xticks(years_list)
axis[1].tick_params(axis="x", rotation=45)

axis[2].plot(years_list, [cf_mv_by_year[y]    for y in years_list],
             marker="o", label="Min-Variance Portfolio",   color="blue")
axis[2].plot(years_list, [cf_mv_50_by_year[y] for y in years_list],
             marker="s", label="Min-Variance (50% CF)",    color="green")
axis[2].plot(years_list, [cf_vw_by_year[y]    for y in years_list],
             marker="o", label="Value-Weighted Portfolio", color="red")
axis[2].plot(years_list, [cf_vw_50_by_year[y] for y in years_list],
             marker="s", label="Value-Weighted (50% CF)",  color="orange")
axis[2].set_title("Carbon Footprint over Time (4 portfolios)")
axis[2].set_xlabel("Year")
axis[2].set_ylabel("Carbon Footprint (tCO₂ / MUSD invested)")
axis[2].legend(fontsize=9)
axis[2].set_xticks(years_list)
axis[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

### 11.3 Trade-off discussion
Pourquoi : la section 3.4 du projet demande de commenter le trade-off entre performance financière et réduction carbone, et d’expliquer la différence entre P
oos
mv
	​

 et P
oos
mv
	​

(0.5), puis entre P
oos
vw
	​

 et P
oos
vw
	​

(0.5). Donc il faut une vraie cellule d’interprétation, pas seulement des figures

In [ ]:
def perf_stats(r):
    mu  = r.mean() * 12
    vol = r.std()  * np.sqrt(12)
    rf_ = rf["rf"].reindex(r.index).fillna(0).mean() * 12
    sr  = (mu - rf_) / vol
    cum = (1 + r).prod() - 1
    return mu, vol, sr, cum

mu_mv, vol_mv,   sr_mv,   cum_mv   = perf_stats(returns_mv)
mu_mv50, vol_mv50, sr_mv50, cum_mv50 = perf_stats(returns_mv_50)
mu_vw, vol_vw,   sr_vw,   cum_vw   = perf_stats(returns_vw)
mu_vw50, vol_vw50, sr_vw50, cum_vw50 = perf_stats(returns_vw_50)

avg_cf_mv = np.mean(list(cf_mv_by_year.values()))
avg_cf_mv50 = np.mean(list(cf_mv_50_by_year.values()))
avg_cf_vw = np.mean(list(cf_vw_by_year.values()))
avg_cf_vw50 = np.mean(list(cf_vw_50_by_year.values()))

print("Cost of the 50% carbon footprint reduction")
print("-" * 60)

print("\n--- Active investor (minimum-variance) ---")
print(f"  Annual return : {mu_mv:>7.2%}  →  {mu_mv50:>7.2%}   "
      f"(Δ = {(mu_mv50 - mu_mv)*10000:+.0f} bps)")
print(f"  Volatility    : {vol_mv:>7.2%}  →  {vol_mv50:>7.2%}   "
      f"(Δ = {(vol_mv50 - vol_mv)*10000:+.0f} bps)")
print(f"  Sharpe ratio  : {sr_mv:>7.3f}  →  {sr_mv50:>7.3f}")
print(f"  Cumul. return : {cum_mv:>7.2%}  →  {cum_mv50:>7.2%}")
print(f"  Average CF    : {avg_cf_mv:>7.1f}  →  {avg_cf_mv50:>7.1f}   "
      f"(−{(1 - avg_cf_mv50/avg_cf_mv)*100:.1f}%)")

print("\n--- Passive investor (value-weighted tracking) ---")
print(f"  Annual return : {mu_vw:>7.2%}  →  {mu_vw50:>7.2%}   "
      f"(Δ = {(mu_vw50 - mu_vw)*10000:+.0f} bps)")
print(f"  Volatility    : {vol_vw:>7.2%}  →  {vol_vw50:>7.2%}   "
      f"(Δ = {(vol_vw50 - vol_vw)*10000:+.0f} bps)")
print(f"  Sharpe ratio  : {sr_vw:>7.3f}  →  {sr_vw50:>7.3f}")
print(f"  Cumul. return : {cum_vw:>7.2%}  →  {cum_vw50:>7.2%}")
print(f"  Average CF    : {avg_cf_vw:>7.1f}  →  {avg_cf_vw50:>7.1f}   "
      f"(−{(1 - avg_cf_vw50/avg_cf_vw)*100:.1f}%)")

te_vw50 = (returns_vw_50 - returns_vw).std() * np.sqrt(12)
print(f"\n  Realised tracking error P(vw)(0.5) vs P(vw): {te_vw50:.2%}")

### 11.4 Portfolio composition analysis
- top 10 weights in P(mv)
- top 10 weights in P(mv)(0.5)
- biggest weight changes
- firms removed or strongly underweighted due to carbon constraint
- if sector data available: sector exposure changes

In [ ]:
company_names = static_data.set_index("ISIN")["NAME"]

def avg_weights(weights_by_year):
    # union of all ISINs across years, then average
    all_isins = list(set().union(*[set(w.index) for w in weights_by_year.values()]))
    total = pd.Series(0.0, index=all_isins)
    for w in weights_by_year.values():
        total = total.add(w.reindex(all_isins).fillna(0.0), fill_value=0.0)
    return total / len(weights_by_year)


avg_w_mv   = avg_weights(mv_weights_by_year)
avg_w_mv50 = avg_weights(mv_50_weights_by_year)
avg_w_vw   = avg_weights(vw_weights_by_year)
avg_w_vw50 = avg_weights(vw_50_weights_by_year)

def print_weight_changes(w1, w2, label):
    isins = w1.index.union(w2.index)
    a = w1.reindex(isins).fillna(0.0)
    b = w2.reindex(isins).fillna(0.0)
    delta = (b - a).sort_values()

    print(f"\n{label}")
    print(f"\nMost reduced (carbon-heavy firms):")
    for isin, d in delta.head(10).items():
        name = company_names.get(isin, "N/A")[:40]
        print(f"  {isin}  {name:<42}  {a[isin]:>6.2%} → {b[isin]:>6.2%}  (Δ {d:+.2%})")

    print(f"\nMost increased (lower CI firms):")
    for isin, d in delta.tail(10)[::-1].items():
        name = company_names.get(isin, "N/A")[:40]
        print(f"  {isin}  {name:<42}  {a[isin]:>6.2%} → {b[isin]:>6.2%}  (Δ {d:+.2%})")

print_weight_changes(avg_w_mv,  avg_w_mv50, "Min-Variance → Min-Variance (50% CF)")
print_weight_changes(avg_w_vw,  avg_w_vw50, "Value-Weighted → Value-Weighted (50% CF)")

# PART III — NET ZERO

## 12. Net Zero Portfolio

### 12.1 Net Zero target and optimization
la consigne 4.1 demande de définir la cible Net Zero (1−θ)

puis de résoudre l’optimisation, puis de calculer les caractéristiques du portefeuille. La référence carbone et l’optimisation vont naturellement ensemble.

In [ ]:
theta = net_zero_reduction_rate
base_year = first_allocation_year

cf_vw_reference = cf_vw_by_year[base_year]

nz_cf_caps = {}
for year in allocation_years:
    nz_cf_caps[year] = (1 - theta) ** (year - base_year + 1) * cf_vw_reference

print(f"Net Zero trajectory — reference CF(vw) 2013 = {cf_vw_reference:.1f}\n")
print(f"{'Year':<6}{'CF cap':>10}{'Reduction':>12}")
print("-" * 30)
for year in allocation_years:
    red = 1 - nz_cf_caps[year] / cf_vw_reference
    print(f"{year:<6}{nz_cf_caps[year]:>10.2f}{red:>11.1%}")

nz_weights_by_year = {}

for year in allocation_years:
    inv_set = investment_set_by_year[year]
    _, sigma = estimated_moments[year]
    co2_year = co2_scope1_annual.loc[co2_scope1_annual.index.year == year].iloc[0]
    cap_year = market_cap_a.loc[market_cap_a.index.year == year].iloc[0]
    w_vw = vw_weights_by_year[year].reindex(inv_set).fillna(0.0)
    cf_cap = nz_cf_caps[year]

    w_opt = optimize_tracking_error_with_cf_cap(
        sigma, w_vw, co2_year, cap_year, cf_cap
    )
    nz_weights_by_year[year] = pd.Series(w_opt, index=inv_set)

    n_active   = (nz_weights_by_year[year] > 1e-4).sum()
    top_weight = nz_weights_by_year[year].max()
    print(f"Year {year}: {len(inv_set)} firms | {n_active} invested | "
          f"max = {top_weight:.2%} | cf_cap = {cf_cap:.1f}")

### 12.2 Out-of-sample returns

In [ ]:
returns_nz = compute_portfolio_returns(monthly_returns, nz_weights_by_year, allocation_years)

print(f"Period : {returns_nz.index[0].date()} → {returns_nz.index[-1].date()}")
print(f"Months : {len(returns_nz)}  (expected: 144)")
print(f"Mean   : {returns_nz.mean():.4%}")
print(f"Vol    : {returns_nz.std():.4%}")
print(f"Min    : {returns_nz.min():.4%}")
print(f"Max    : {returns_nz.max():.4%}")

### 12.3 Performance, WACI and Carbon Footprint

In [ ]:
stats_nz = compute_performance_stats(returns_nz, rf["rf"], "Net Zero (10%/year)")

stats_compare_nz = pd.DataFrame([stats_vw, stats_vw_50, stats_nz]).set_index("Portfolio")
print("Performance summary")
print(stats_compare_nz.to_string())

waci_nz_by_year = {}
cf_nz_by_year = {}

for year in allocation_years:
    ci_year = carbon_intensity.loc[carbon_intensity.index.year == year].iloc[0]
    co2_year = co2_scope1_annual.loc[co2_scope1_annual.index.year == year].iloc[0]
    cap_year = market_cap_a.loc[market_cap_a.index.year == year].iloc[0]
    w = nz_weights_by_year[year]

    waci_nz_by_year[year] = compute_waci(w, ci_year)
    cf_nz_by_year[year] = compute_carbon_footprint(w, co2_year, cap_year, portfolio_value)

    cap_value = nz_cf_caps[year]

years_list = sorted(waci_vw_by_year.keys())

cum_vw = (1 + returns_vw).cumprod()
cum_vw_50 = (1 + returns_vw_50).cumprod()
cum_nz = (1 + returns_nz).cumprod()

fig, axis = plt.subplots(1, 3, figsize=(18, 5))

axis[0].plot(years_list, [waci_vw_by_year[y] for y in years_list],
             marker="o", label="Value-Weighted Portfolio",  color="red", linewidth=1.8)
axis[0].plot(years_list, [waci_vw_50_by_year[y] for y in years_list],
             marker="s", label="Value-Weighted (50% CF)",   color="green", linewidth=1.8)
axis[0].plot(years_list, [waci_nz_by_year[y] for y in years_list],
             marker="^", label="Net Zero Portfolio", color="purple",linewidth=1.8)
axis[0].set_title("WACI over Time")
axis[0].set_xlabel("Year")
axis[0].set_ylabel("WACI (tCO₂ / MUSD revenue)")
axis[0].legend()
axis[0].set_xticks(years_list)
axis[0].tick_params(axis="x", rotation=45)

axis[1].plot(years_list, [cf_vw_by_year[y] for y in years_list],
             marker="o", label="Value-Weighted Portfolio", color="red", linewidth=1.8)
axis[1].plot(years_list, [cf_vw_50_by_year[y] for y in years_list],
             marker="s", label="Value-Weighted (50% CF)", color="green", linewidth=1.8)
axis[1].plot(years_list, [cf_nz_by_year[y] for y in years_list],
             marker="^", label="Net Zero Portfolio", color="purple",linewidth=1.8)
axis[1].plot(years_list, [nz_cf_caps[y] for y in years_list], linestyle=":", color="gray", linewidth=1.4, label="NZ target")
axis[1].set_title("Carbon Footprint over Time")
axis[1].set_xlabel("Year")
axis[1].set_ylabel("Carbon Footprint (tCO₂ / MUSD invested)")
axis[1].legend()
axis[1].set_xticks(years_list)
axis[1].tick_params(axis="x", rotation=45)

# cumulative returns
axis[2].plot(cum_vw.index, cum_vw.values, label="Value-Weighted Portfolio", color="red", linewidth=1.8)
axis[2].plot(cum_vw_50.index, cum_vw_50.values, label="Value-Weighted (50% CF)", color="green", linewidth=1.8)
axis[2].plot(cum_nz.index,cum_nz.values, label="Net Zero Portfolio", color="purple", linewidth=1.8)
axis[2].axhline(y=1, color="black", linestyle="--", linewidth=0.8, alpha=0.5)
axis[2].set_title("Cumulative Performance")
axis[2].set_xlabel("Date")
axis[2].set_ylabel("Cumulative performance (base 1)")
axis[2].legend()
axis[2].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
axis[2].xaxis.set_major_locator(mdates.YearLocator())
axis[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

te_nz = (returns_nz - returns_vw).std() * np.sqrt(12)
print(f"\nRealised tracking error VW(NZ) vs VW: {te_nz:.2%}")

## 13. Final Comparison

### 13.1 Summary table

In [ ]:
summary_df_nz = pd.DataFrame([stats_vw, stats_vw_50, stats_nz]).set_index("Portfolio")

te_50 = (returns_vw_50 - returns_vw).std() * np.sqrt(12)
te_nz = (returns_nz - returns_vw).std() * np.sqrt(12)

portfolios_nz = [
    (returns_vw, waci_vw_by_year, cf_vw_by_year, 0.0),
    (returns_vw_50, waci_vw_50_by_year, cf_vw_50_by_year, te_50),
    (returns_nz, waci_nz_by_year, cf_nz_by_year, te_nz),]

summary_df_nz["Cumul. Return"] = [f"{cum_ret(r):.2%}" for r, _, _, _ in portfolios_nz]
summary_df_nz["Avg WACI"] = [f"{np.mean(list(w.values())):.1f}" for _, w, _, _ in portfolios_nz]
summary_df_nz["Avg CF"] = [f"{np.mean(list(c.values())):.1f}" for _, _, c, _ in portfolios_nz]
summary_df_nz["TE vs VW"] = [f"{te:.2%}" for _, _, _, te in portfolios_nz]
print(summary_df_nz.to_string())

### 13.2 Cumulative performance and carbon footprint comparison
Cumulative returns plot, CF evolution plot

In [ ]:
years_list = sorted(cf_vw_by_year.keys())

cum_vw = (1 + returns_vw).cumprod()
cum_vw_50 = (1 + returns_vw_50).cumprod()
cum_nz = (1 + returns_nz).cumprod()

fig, axis = plt.subplots(1, 2, figsize=(14, 5))

axis[0].plot(cum_vw.index, cum_vw.values, label="Value-Weighted Portfolio", color="red",  linewidth=1.8)
axis[0].plot(cum_vw_50.index, cum_vw_50.values, marker="^", label="Value-Weighted (50% CF)",  color="green", linewidth=1.8)
axis[0].plot(cum_nz.index, cum_nz.values, label="Net Zero Portfolio", color="purple", linewidth=1.8)
axis[0].axhline(y=1, color="black", linestyle="--", linewidth=0.8, alpha=0.5)
axis[0].set_title("Cumulative Performance")
axis[0].set_xlabel("Date")
axis[0].set_ylabel("Cumulative performance")
axis[0].legend()
axis[0].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
axis[0].xaxis.set_major_locator(mdates.YearLocator())
axis[0].tick_params(axis="x", rotation=45)

axis[1].plot(years_list, [cf_vw_by_year[y] for y in years_list], marker="o", label="Value-Weighted Portfolio", color="red", linewidth=1.8)
axis[1].plot(years_list, [cf_vw_50_by_year[y] for y in years_list], marker="s", label="Value-Weighted (50% CF)", color="green", linewidth=1.8)
axis[1].plot(years_list, [cf_nz_by_year[y] for y in years_list], marker="^", label="Net Zero Portfolio", color="purple", linewidth=1.8)
axis[1].plot(years_list, [0.5 * cf_vw_by_year[y] for y in years_list], linestyle=":", color="black", alpha=0.5, label="50% target")
axis[1].plot(years_list, [nz_cf_caps[y] for y in years_list], linestyle=":", color="gray", alpha=0.5, label="NZ target")
axis[1].set_title("Carbon Footprint over Time")
axis[1].set_xlabel("Year")
axis[1].set_ylabel("Carbon Footprint (tCO₂ / MUSD invested)")
axis[1].legend()
axis[1].set_xticks(years_list)
axis[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

### 13.3 Cost of the Net Zero strategy

In [ ]:
def show_cost(label, r, cf_dict, r_base, cf_base):
    mu     = r.mean() * 12
    vol    = r.std()  * np.sqrt(12)
    rf_    = rf["rf"].reindex(r.index).fillna(0).mean() * 12
    sr     = (mu - rf_) / vol
    cum    = (1 + r).prod() - 1
    avg_cf = np.mean(list(cf_dict.values()))

    mu_base     = r_base.mean() * 12
    cum_base    = (1 + r_base).prod() - 1
    avg_cf_base = np.mean(list(cf_base.values()))

    print(f"\n--- {label} ---")
    print(f"  Annual return : {mu:>7.2%}   (Δ vs VW = {(mu - mu_base)*10000:+.0f} bps)")
    print(f"  Volatility    : {vol:>7.2%}")
    print(f"  Sharpe        : {sr:>7.3f}")
    print(f"  Cumul. return : {cum:>7.2%}   (Δ vs VW = {(cum - cum_base)*100:+.2f} pp)")
    print(f"  Average CF    : {avg_cf:>7.1f}   (−{(1 - avg_cf/avg_cf_base)*100:.1f}% vs VW)")

show_cost("VW (benchmark)", returns_vw,    cf_vw_by_year,    returns_vw, cf_vw_by_year)
show_cost("VW(0.5)",        returns_vw_50, cf_vw_50_by_year, returns_vw, cf_vw_by_year)
show_cost("VW(NZ)",         returns_nz,    cf_nz_by_year,    returns_vw, cf_vw_by_year)

# is the NZ constraint binding?
print("\nNet Zero constraint — binding check")
print("-" * 55)
print(f"{'Year':<6}{'CF realised':>14}{'NZ cap':>12}{'Slack':>12}{'Status':>10}")
print("-" * 55)
for year in allocation_years:
    cf_r   = cf_nz_by_year[year]
    cap    = nz_cf_caps[year]
    slack  = cap - cf_r
    status = "BINDING" if abs(slack) < 1.0 else "slack"
    print(f"{year:<6}{cf_r:>14.2f}{cap:>12.2f}{slack:>12.2f}{status:>10}")

# key numbers
cf_red_50 = (1 - np.mean(list(cf_vw_50_by_year.values())) / np.mean(list(cf_vw_by_year.values()))) * 100
cf_red_nz = (1 - np.mean(list(cf_nz_by_year.values()))    / np.mean(list(cf_vw_by_year.values()))) * 100
delta_50  = (returns_vw_50.mean() - returns_vw.mean()) * 12 * 10000
delta_nz  = (returns_nz.mean()    - returns_vw.mean()) * 12 * 10000

print(f"\nVW(0.5): CF −{cf_red_50:.1f}% vs VW | cost = {delta_50:+.0f} bps/year | TE = {te_50*100:.2f}%")
print(f"VW(NZ) : CF −{cf_red_nz:.1f}% vs VW | cost = {delta_nz:+.0f} bps/year | TE = {te_nz*100:.2f}%")

#Robustness Check

In [ ]:
print("Robustness checks — investment set size (avg over 2013-2024)")
print("=" * 65)
print(f"  {'Parametrization':<38} | {'Avg N':>7} | {'Min N':>7} | {'Max N':>7}")
print("-" * 65)

def inv_set_sizes(stale_thresh, min_obs, ffill_lim):
    ri_rb = ri_monthly.ffill(limit=ffill_lim) if ffill_lim else ri_monthly.ffill()
    for isin in ri_rb.columns:
        fv = first_obs[isin]
        if not pd.isna(fv):
            ri_rb.loc[ri_rb.index < fv, isin] = np.nan
    ret_rb = ri_rb.pct_change(fill_method=None).iloc[1:].loc[:last_ret_date]
    sizes = []
    for year in allocation_years:
        inv = build_investment_set(year, regional_isins, ret_rb,
                                   co2_scope1_annual, ri_rb,
                                   lookback_months=estimation_window_months,
                                   minimum_months=min_obs,
                                   max_zero_share=stale_thresh)
        sizes.append(len(inv))
    return sizes

def show(label, sizes):
    print(f"  {label:<38} | {np.mean(sizes):>7.0f} | "
          f"{np.min(sizes):>7.0f} | {np.max(sizes):>7.0f}")

show("Baseline (stale=50%, min=36m, ffill=2)",
     [len(v) for v in investment_set_by_year.values()])
print("-" * 65)
show("Stale threshold = 30%",   inv_set_sizes(0.30, 36, 2))
show("Stale threshold = 70%",   inv_set_sizes(0.70, 36, 2))
print("-" * 65)
show("Min observations = 24m",  inv_set_sizes(0.50, 24, 2))
show("Min observations = 60m",  inv_set_sizes(0.50, 60, 2))
print("-" * 65)
show("Forward-fill limit = 1m", inv_set_sizes(0.50, 36, 1))
show("Forward-fill no limit",   inv_set_sizes(0.50, 36, None))
print("=" * 65)


# --- Carbon footprint comparison: ffill=2 vs ffill=no limit ---
# The only case where results differ meaningfully is the ffill no-limit,
# which inflates CF in 2024 due to zombie firms like SAS.

def get_cf_mv(ffill_lim):
    ri_rb = ri_monthly.ffill(limit=ffill_lim) if ffill_lim else ri_monthly.ffill()
    for isin in ri_rb.columns:
        fv = first_obs[isin]
        if not pd.isna(fv):
            ri_rb.loc[ri_rb.index < fv, isin] = np.nan
    ret_rb = ri_rb.pct_change(fill_method=None).iloc[1:].loc[:last_ret_date]

    inv_sets_rb, weights_rb = {}, {}
    for year in allocation_years:
        inv_sets_rb[year] = build_investment_set(
            year, regional_isins, ret_rb, co2_scope1_annual, ri_rb,
            lookback_months=estimation_window_months,
            minimum_months=minimum_valid_months,
            max_zero_share=maximum_zero_return_share
        )
        _, sigma = estimate_moments(ret_rb, inv_sets_rb[year],
                                    ret_rb.index[ret_rb.index.year == year][-1])
        w = optimize_min_variance(sigma)
        weights_rb[year] = pd.Series(w, index=inv_sets_rb[year])

    cf_by_year_rb = {}
    for year in allocation_years:
        co2_year = co2_scope1_annual.loc[co2_scope1_annual.index.year == year].iloc[0]
        cap_year = market_cap_a.loc[market_cap_a.index.year == year].iloc[0]
        cf_by_year_rb[year] = compute_carbon_footprint(
            weights_rb[year], co2_year, cap_year, portfolio_value
        )
    return cf_by_year_rb

print("\nComputing CF for ffill alternatives (takes ~2 min)...")
cf_ffill2    = cf_mv_by_year          # already computed — baseline
cf_ffill1    = get_cf_mv(ffill_lim=1)
cf_ffill_inf = get_cf_mv(ffill_lim=None)

# plot
years_list = sorted(allocation_years)
fig, axis = plt.subplots(figsize=(10, 5))

axis.plot(years_list, [cf_ffill2[y]    for y in years_list],
          marker="o", label="Baseline (ffill=2)", color="steelblue", linewidth=1.8)
axis.plot(years_list, [cf_ffill1[y]    for y in years_list],
          marker="s", label="ffill = 1 month",    color="green",     linewidth=1.8,
          linestyle="--")
axis.plot(years_list, [cf_ffill_inf[y] for y in years_list],
          marker="^", label="ffill no limit",      color="red",       linewidth=1.8,
          linestyle=":")

axis.set_title("Carbon Footprint — sensitivity to forward-fill limit\n"
               "P(mv)_oos, Group AI (AMER+EUR, Scope 1)")
axis.set_xlabel("Year")
axis.set_ylabel("Carbon Footprint (tCO₂ / MUSD invested)")
axis.legend()
axis.set_xticks(years_list)
axis.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

print("\nffill=1 and ffill=2 produce identical CF — choice of limit has no impact.")
print("ffill no-limit inflates CF in 2024 due to zombie firms (e.g., SAS).")
print("This validates our choice of ffill(limit=2).")